# Assignment 4 — Optimizing Transformer Translation with Ray Tune & Optuna
**Roll No:** B23CM1060

This notebook implements Part 2 (Ray Tune + Optuna sweep) and Part 3 (efficiency challenge).
The baseline (Part 1) was already completed separately.

---

In [28]:
!pip install ray[tune] optuna -q
!pip install nltk   

In [29]:
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
import math
import os
import pickle
import time
from collections import Counter
from torch.utils.data import Dataset, DataLoader

import ray
from ray import tune
from ray.tune.search.optuna import OptunaSearch
from ray.tune.schedulers import ASHAScheduler

import nltk
from nltk.translate.bleu_score import corpus_bleu, SmoothingFunction
nltk.download('punkt', quiet=True)

print('All imports successful.')
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')

All imports successful.
Device: cuda


In [30]:
df = pd.read_csv(
    'English-Hindi.tsv',
    sep='\t', header=None, names=['id1', 'en', 'id2', 'hi']
)
df = df[['en', 'hi']].dropna().reset_index(drop=True)
print(f'Total pairs: {len(df)}')
df.head(3)

Total pairs: 13186


,en,hi
0,Muiriel is 20 now.,म्यूरियल अब बीस साल की हो गई है।
1,Muiriel is 20 now.,म्यूरियल अब बीस साल की है।
2,Education in this world disappoints me.,मैं इस दुनिया में शिक्षा पर बहुत निराश हूँ।


In [31]:
class Vocabulary:
    def __init__(self, freq_threshold=2):
        self.freq_threshold = freq_threshold
        self.itos = {0: '<pad>', 1: '<sos>', 2: '<eos>', 3: '<unk>'}
        self.stoi = {'<pad>': 0, '<sos>': 1, '<eos>': 2, '<unk>': 3}
        self.idx = 4

    def build_vocab(self, sentence_list):
        frequencies = Counter()
        for sentence in sentence_list:
            for word in self.tokenize(sentence):
                frequencies[word] += 1
        for word, freq in frequencies.items():
            if freq >= self.freq_threshold:
                self.stoi[word] = self.idx
                self.itos[self.idx] = word
                self.idx += 1

    def tokenize(self, sentence):
        return sentence.lower().strip().split()

    def numericalize(self, sentence):
        tokens = self.tokenize(sentence)
        return [self.stoi.get(t, self.stoi['<unk>']) for t in tokens]

    def __len__(self):
        return len(self.stoi)

    def __getitem__(self, token):
        return self.stoi.get(token, self.stoi['<unk>'])


en_vocab = Vocabulary(freq_threshold=2)
hi_vocab = Vocabulary(freq_threshold=2)
en_vocab.build_vocab(df['en'].tolist())
hi_vocab.build_vocab(df['hi'].tolist())

print(f'English vocab size: {len(en_vocab)}')
print(f'Hindi vocab size:   {len(hi_vocab)}')

English vocab size: 4117
Hindi vocab size:   4044


In [32]:
MAX_LEN = 50
SRC_PAD_IDX = en_vocab['<pad>']
TGT_PAD_IDX = hi_vocab['<pad>']


def encode_sentence(sentence, vocab, max_len=50):
    tokens = [vocab.stoi['<sos>']] + vocab.numericalize(sentence)[:max_len - 2] + [vocab.stoi['<eos>']]
    return tokens + [vocab.stoi['<pad>']] * (max_len - len(tokens))


class TranslationDataset(Dataset):
    def __init__(self, dataframe, en_vocab, hi_vocab, max_len=50):
        self.en_sentences = dataframe['en'].tolist()
        self.hi_sentences = dataframe['hi'].tolist()
        self.en_vocab = en_vocab
        self.hi_vocab = hi_vocab
        self.max_len = max_len

    def __len__(self):
        return len(self.en_sentences)

    def __getitem__(self, idx):
        src = encode_sentence(self.en_sentences[idx], self.en_vocab, self.max_len)
        tgt = encode_sentence(self.hi_sentences[idx], self.hi_vocab, self.max_len)
        return torch.tensor(src), torch.tensor(tgt)


def collate_fn(batch):
    src_batch, tgt_batch = zip(*batch)
    src_batch = torch.stack(src_batch)
    tgt_batch = torch.stack(tgt_batch)
    # Teacher forcing split: input = tgt[:-1], label = tgt[1:]
    tgt_input  = tgt_batch[:, :-1]
    tgt_output = tgt_batch[:, 1:]
    return src_batch, tgt_input, tgt_output


# Global dataset (shared across all Ray workers via globals)
train_dataset = TranslationDataset(df, en_vocab, hi_vocab, max_len=MAX_LEN)
src_vocab_size = len(en_vocab)
tgt_vocab_size = len(hi_vocab)
print(f'Dataset size: {len(train_dataset)}')

Dataset size: 13186


In [33]:
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=5000):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len).unsqueeze(1).float()
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        pe = pe.unsqueeze(0)
        self.register_buffer('pe', pe)

    def forward(self, x):
        return x + self.pe[:, :x.size(1)]


class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, num_heads, dropout=0.1):
        super().__init__()
        assert d_model % num_heads == 0, 'd_model must be divisible by num_heads'
        self.d_model = d_model
        self.num_heads = num_heads
        self.d_k = d_model // num_heads
        self.query_linear = nn.Linear(d_model, d_model)
        self.key_linear   = nn.Linear(d_model, d_model)
        self.value_linear = nn.Linear(d_model, d_model)
        self.out_linear   = nn.Linear(d_model, d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, q, k, v, mask=None):
        B = q.size(0)
        Q = self.query_linear(q).view(B, -1, self.num_heads, self.d_k).transpose(1, 2)
        K = self.key_linear(k).view(B, -1, self.num_heads, self.d_k).transpose(1, 2)
        V = self.value_linear(v).view(B, -1, self.num_heads, self.d_k).transpose(1, 2)
        scores = torch.matmul(Q, K.transpose(-2, -1)) / (self.d_k ** 0.5)
        if mask is not None:
            scores = scores.masked_fill(mask == 0, -1e9)
        attn = self.dropout(torch.softmax(scores, dim=-1))
        out = torch.matmul(attn, V).transpose(1, 2).contiguous().view(B, -1, self.d_model)
        return self.out_linear(out)


class FeedForward(nn.Module):
    def __init__(self, d_model, d_ff=2048, dropout=0.1):
        super().__init__()
        self.linear1 = nn.Linear(d_model, d_ff)
        self.linear2 = nn.Linear(d_ff, d_model)
        self.dropout = nn.Dropout(dropout)
        self.relu = nn.ReLU()

    def forward(self, x):
        return self.linear2(self.dropout(self.relu(self.linear1(x))))


class LayerNorm(nn.Module):
    def __init__(self, d_model, eps=1e-6):
        super().__init__()
        self.gamma = nn.Parameter(torch.ones(d_model))
        self.beta  = nn.Parameter(torch.zeros(d_model))
        self.eps   = eps

    def forward(self, x):
        mean = x.mean(-1, keepdim=True)
        std  = x.std(-1, keepdim=True)
        return self.gamma * (x - mean) / (std + self.eps) + self.beta


class EncoderLayer(nn.Module):
    def __init__(self, d_model, num_heads, d_ff, dropout=0.1):
        super().__init__()
        self.self_attn = MultiHeadAttention(d_model, num_heads, dropout)
        self.ffn   = FeedForward(d_model, d_ff, dropout)
        self.norm1 = LayerNorm(d_model)
        self.norm2 = LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, mask=None):
        x = self.norm1(x + self.dropout(self.self_attn(x, x, x, mask)))
        x = self.norm2(x + self.dropout(self.ffn(x)))
        return x


class DecoderLayer(nn.Module):
    def __init__(self, d_model, num_heads, d_ff, dropout=0.1):
        super().__init__()
        self.self_attn  = MultiHeadAttention(d_model, num_heads, dropout)
        self.cross_attn = MultiHeadAttention(d_model, num_heads, dropout)
        self.ffn   = FeedForward(d_model, d_ff, dropout)
        self.norm1 = LayerNorm(d_model)
        self.norm2 = LayerNorm(d_model)
        self.norm3 = LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, enc_out, src_mask=None, tgt_mask=None):
        x = self.norm1(x + self.dropout(self.self_attn(x, x, x, tgt_mask)))
        x = self.norm2(x + self.dropout(self.cross_attn(x, enc_out, enc_out, src_mask)))
        x = self.norm3(x + self.dropout(self.ffn(x)))
        return x


class Encoder(nn.Module):
    def __init__(self, input_vocab_size, d_model, num_layers, num_heads, d_ff, max_len, dropout=0.1):
        super().__init__()
        self.embed   = nn.Embedding(input_vocab_size, d_model)
        self.pos_enc = PositionalEncoding(d_model, max_len)
        self.layers  = nn.ModuleList([EncoderLayer(d_model, num_heads, d_ff, dropout) for _ in range(num_layers)])
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, mask=None):
        x = self.dropout(self.pos_enc(self.embed(x)))
        for layer in self.layers:
            x = layer(x, mask)
        return x


class Decoder(nn.Module):
    def __init__(self, target_vocab_size, d_model, num_layers, num_heads, d_ff, max_len, dropout=0.1):
        super().__init__()
        self.embed   = nn.Embedding(target_vocab_size, d_model)
        self.pos_enc = PositionalEncoding(d_model, max_len)
        self.layers  = nn.ModuleList([DecoderLayer(d_model, num_heads, d_ff, dropout) for _ in range(num_layers)])
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, enc_out, src_mask=None, tgt_mask=None):
        x = self.dropout(self.pos_enc(self.embed(x)))
        for layer in self.layers:
            x = layer(x, enc_out, src_mask, tgt_mask)
        return x


class Transformer(nn.Module):
    def __init__(self, src_vocab_size, tgt_vocab_size, d_model=512, num_layers=6,
                 num_heads=8, d_ff=2048, max_len=100, dropout=0.1):
        super().__init__()
        self.encoder = Encoder(src_vocab_size, d_model, num_layers, num_heads, d_ff, max_len, dropout)
        self.decoder = Decoder(tgt_vocab_size, d_model, num_layers, num_heads, d_ff, max_len, dropout)
        self.fc_out  = nn.Linear(d_model, tgt_vocab_size)

    def make_pad_mask(self, seq, pad_idx):
        return (seq != pad_idx).unsqueeze(1).unsqueeze(2)

    def make_subsequent_mask(self, size):
        return torch.tril(torch.ones((size, size))).bool().to(next(self.parameters()).device)

    def forward(self, src, tgt, src_pad_idx, tgt_pad_idx):
        src_mask     = self.make_pad_mask(src, src_pad_idx)
        tgt_pad_mask = self.make_pad_mask(tgt, tgt_pad_idx)
        tgt_sub_mask = self.make_subsequent_mask(tgt.size(1))
        tgt_mask     = tgt_pad_mask & tgt_sub_mask
        enc_out = self.encoder(src, src_mask)
        dec_out = self.decoder(tgt, enc_out, src_mask, tgt_mask)
        return self.fc_out(dec_out)

In [34]:
def train_tune(config):
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

    model = Transformer(
        src_vocab_size=src_vocab_size,
        tgt_vocab_size=tgt_vocab_size,
        d_model=512,                     
        num_layers=6,                     
        num_heads=config['num_heads'],
        d_ff=config['d_ff'],
        max_len=MAX_LEN,
        dropout=config['dropout']
    ).to(device)

    if config.get('optimizer', 'adam') == 'adamw':
        optimizer = torch.optim.AdamW(
            model.parameters(), lr=config['lr'],
            weight_decay=config.get('weight_decay', 1e-4)
        )
    else:
        optimizer = torch.optim.Adam(model.parameters(), lr=config['lr'])

    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer, T_max=config['epochs']
    )

    criterion = nn.CrossEntropyLoss(ignore_index=TGT_PAD_IDX)

    loader = DataLoader(
        train_dataset,
        batch_size=config['batch_size'],
        shuffle=True,
        collate_fn=collate_fn,
        num_workers=2,
        pin_memory=(device.type == 'cuda')
    )

    for epoch in range(config['epochs']):
        model.train()
        total_loss = 0.0

        for src, tgt_input, tgt_output in loader:
            src        = src.to(device)
            tgt_input  = tgt_input.to(device)
            tgt_output = tgt_output.to(device)

            optimizer.zero_grad()

            output = model(src, tgt_input, SRC_PAD_IDX, TGT_PAD_IDX)

            output_flat = output.reshape(-1, output.shape[-1])
            target_flat = tgt_output.reshape(-1)

            loss = criterion(output_flat, target_flat)
            loss.backward()

            nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)

            optimizer.step()
            total_loss += loss.item()

        scheduler.step()
        avg_loss = total_loss / len(loader)

        tune.report({'loss': avg_loss, 'epoch': epoch + 1})

In [35]:
search_space = {
    'lr':          tune.loguniform(1e-5, 1e-2),     
    'batch_size':  tune.choice([16, 32, 64,128]),      
    'num_heads':   tune.choice([4, 8, 16]),           
    'd_ff':        tune.choice([1024, 2048]),       
    'dropout':     tune.uniform(0.1, 0.4),    
    'optimizer':   tune.choice(['adam', 'adamw']), 
    'weight_decay': tune.loguniform(1e-5, 1e-2),    
    'epochs':      30,                             
    }

asha_scheduler = ASHAScheduler(
    metric='loss',
    mode='min',
    max_t=20,       
    grace_period=5,   
    reduction_factor=2 
)

optuna_search = OptunaSearch(metric='loss', mode='min')

In [ ]:
if not ray.is_initialized():
    ray.init(ignore_reinit_error=True, num_gpus=1)

sweep_start = time.time()

tuner = tune.Tuner(
    tune.with_resources(
        train_tune,
        resources={'gpu': 1, 'cpu': 8}
    ),
    tune_config=tune.TuneConfig(
        # metric = "loss",
        search_alg=optuna_search,
        scheduler=asha_scheduler,
        num_samples=20,
        verbose=0
    ),
    param_space=search_space, 
)

results = tuner.fit()

sweep_duration = time.time() - sweep_start
print(f'\nSweep finished in {sweep_duration/60:.1f} min')

In [37]:
# Retrieve and display best trial
best_result = results.get_best_result(metric='loss', mode='min')
best_config  = best_result.config
best_loss    = best_result.metrics['loss']

print('=' * 55)
print('          BEST TRIAL RESULTS')
print('=' * 55)
for k, v in best_config.items():
    print(f'  {k:20s}: {v}')
print(f'  {"Best Loss":20s}: {best_loss:.4f}')
print('=' * 55)

# Summary table of all trials
results_df = results.get_dataframe()
print('\nAll trial results (sorted by loss):')
summary_cols = ['loss', 'config/lr', 'config/batch_size', 'config/num_heads',
                'config/d_ff', 'config/dropout', 'config/optimizer']
available = [c for c in summary_cols if c in results_df.columns]
print(results_df[available].sort_values('loss').to_string(index=False))

          BEST TRIAL RESULTS
  lr                  : 0.00021199626512112302
  batch_size          : 16
  num_heads           : 16
  d_ff                : 2048
  dropout             : 0.18722688963031334
  optimizer           : adamw
  weight_decay        : 0.0027098820361447875
  epochs              : 30
  Best Loss           : 0.2063

All trial results (sorted by loss):
    loss  config/lr  config/batch_size  config/num_heads  config/d_ff  config/dropout config/optimizer
0.206259   0.000212                 16                16         2048        0.187227            adamw
0.271286   0.000141                 32                16         1024        0.157789             adam
0.524204   0.000296                128                16         1024        0.208805             adam
0.598578   0.000152                 32                16         1024        0.232609            adamw
0.658722   0.000225                128                16         1024        0.187028             adam
1.536780

In [38]:
def retrain_best(config, num_epochs=None, verbose=True):
    """
    Full training run with the best config.
    Returns the trained model and loss history.
    """
    epochs = num_epochs if num_epochs else config['epochs']
    device = DEVICE

    model = Transformer(
        src_vocab_size=src_vocab_size,
        tgt_vocab_size=tgt_vocab_size,
        d_model=512,
        num_layers=6,
        num_heads=config['num_heads'],
        d_ff=config['d_ff'],
        max_len=MAX_LEN,
        dropout=config['dropout']
    ).to(device)

    if config.get('optimizer', 'adam') == 'adamw':
        optimizer = torch.optim.AdamW(
            model.parameters(), lr=config['lr'],
            weight_decay=config.get('weight_decay', 1e-4)
        )
    else:
        optimizer = torch.optim.Adam(model.parameters(), lr=config['lr'])

    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)
    criterion = nn.CrossEntropyLoss(ignore_index=TGT_PAD_IDX)

    loader = DataLoader(
        train_dataset,
        batch_size=config['batch_size'],
        shuffle=True,
        collate_fn=collate_fn,
        num_workers=2,
        pin_memory=(device.type == 'cuda')
    )

    loss_history = []
    t0 = time.time()

    for epoch in range(1, epochs + 1):
        model.train()
        total_loss = 0.0

        for src, tgt_input, tgt_output in loader:
            src        = src.to(device)
            tgt_input  = tgt_input.to(device)
            tgt_output = tgt_output.to(device)

            optimizer.zero_grad()
            output = model(src, tgt_input, SRC_PAD_IDX, TGT_PAD_IDX)
            loss = criterion(output.reshape(-1, output.shape[-1]), tgt_output.reshape(-1))
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            total_loss += loss.item()

        scheduler.step()
        avg_loss = total_loss / len(loader)
        loss_history.append(avg_loss)

        if verbose and (epoch % 5 == 0 or epoch == 1):
            elapsed = (time.time() - t0) / 60
            print(f'  Epoch {epoch:3d}/{epochs} | Loss: {avg_loss:.4f} | Elapsed: {elapsed:.1f} min')

    total_time = time.time() - t0
    print(f'\nTraining complete — {total_time/60:.1f} min | Final Loss: {loss_history[-1]:.4f}')
    return model, loss_history

In [43]:
# Train the best model for 20 epochs (the efficiency target)
print('Retraining with best config for 20 epochs...')
print(f'Config: {best_config}\n')

best_model, loss_history = retrain_best(best_config, num_epochs=30)

Retraining with best config for 20 epochs...
Config: {'lr': 0.00021199626512112302, 'batch_size': 16, 'num_heads': 16, 'd_ff': 2048, 'dropout': 0.18722688963031334, 'optimizer': 'adamw', 'weight_decay': 0.0027098820361447875, 'epochs': 30}

  Epoch   1/30 | Loss: 5.1166 | Elapsed: 1.3 min
  Epoch   5/30 | Loss: 2.3944 | Elapsed: 6.2 min
  Epoch  10/30 | Loss: 0.9663 | Elapsed: 12.1 min
  Epoch  15/30 | Loss: 0.4106 | Elapsed: 16.4 min
  Epoch  20/30 | Loss: 0.2191 | Elapsed: 19.4 min
  Epoch  25/30 | Loss: 0.1363 | Elapsed: 22.4 min
  Epoch  30/30 | Loss: 0.1082 | Elapsed: 25.3 min

Training complete — 25.3 min | Final Loss: 0.1082


In [44]:
def translate_sentence(model, sentence, en_vocab, hi_vocab, max_len=50):
    model.eval()
    tokens = encode_sentence(sentence, en_vocab, max_len=max_len)
    src_tensor = torch.tensor(tokens).unsqueeze(0).to(DEVICE)

    tgt_tokens = [hi_vocab['<sos>']]
    for _ in range(max_len):
        tgt_tensor = torch.tensor(tgt_tokens).unsqueeze(0).to(DEVICE)
        with torch.no_grad():
            output = model(src_tensor, tgt_tensor, SRC_PAD_IDX, TGT_PAD_IDX)
        next_token = output[0, -1].argmax().item()
        tgt_tokens.append(next_token)
        if next_token == hi_vocab['<eos>']:
            break

    return ' '.join(hi_vocab.itos[idx] for idx in tgt_tokens[1:-1])


# Quick translation demo
examples = [
    'I love you.',
    'What is your name?',
    'How are you?',
    'The weather is nice today.',
    'She is a good teacher.',
]
print('Sample translations (best model):')
print('-' * 50)
for s in examples:
    print(f'EN: {s}')
    print(f'HI: {translate_sentence(best_model, s, en_vocab, hi_vocab)}')
    print()

Sample translations (best model):
--------------------------------------------------
EN: I love you.
HI: मैं तुमसे प्यार करता हूँ।

EN: What is your name?
HI: तुम्हारा नाम क्या है?

EN: How are you?
HI: आप कैसे हैं?

EN: The weather is nice today.
HI: आज मौसम अच्छा है।

EN: She is a good teacher.
HI: वह अच्छा शिक्षक है।



In [47]:
smoothie = SmoothingFunction().method4

def evaluate_bleu(model, val_data, en_vocab, hi_vocab, max_len=50):
    references  = []
    hypotheses  = []
    for en_sent, hi_sent in val_data:
        pred        = translate_sentence(model, en_sent, en_vocab, hi_vocab, max_len)
        hypotheses.append(pred.split())
        references.append([hi_sent.split()])
    score = corpus_bleu(references, hypotheses, smoothing_function=smoothie)
    print(f'BLEU Score: {score * 100:.2f}')
    return score


# Validation set (same as baseline)
val_dataset_eval = [
    ('I love you.',                  'मैं तुमसे प्यार करता हूँ।'),
    ('How are you?',                 'आप कैसे हैं?'),
    ('You should sleep.',            'आपको सोना चाहिए।'),
    ('Maybe Tom does not love you.', 'टॉम शायद तुमसे प्यार नहीं करता है।'),
    ('Let me tell Tom.',             'मुझे टॉम को बताने दीजिए।'),
]

print('BLEU evaluation — Best Tuned Model (30 epochs):')
bleu_tuned = evaluate_bleu(best_model, val_dataset_eval, en_vocab, hi_vocab)

BLEU evaluation — Best Tuned Model (30 epochs):
BLEU Score: 90.38


In [46]:
# Save best model weights
model_path = 'b23cm1060_ass_4_best_model.pth'
torch.save(best_model.state_dict(), model_path)
print(f'Best model saved → {model_path}')

# Save vocabs
with open('en_vocab.pkl', 'wb') as f:
    pickle.dump(en_vocab, f)
with open('hi_vocab.pkl', 'wb') as f:
    pickle.dump(hi_vocab, f)
print('Vocabularies saved.')

# Save best config
import json
with open('best_config.json', 'w') as f:
    json.dump(best_config, f, indent=2)
print('Best config saved.')

Best model saved → b23cm1060_ass_4_best_model.pth
Vocabularies saved.
Best config saved.
